In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import time

In [2]:
spark = SparkSession.builder \
    .appName("Ecommerce_Lab") \
    .master("spark://spark-master:7077") \
    .config("spark.ui.port", "4042") \
    .config("spark.executor.instances","2") \
    .config("spark.executor.cores","1") \
    .config("spark.executor.memory","1g") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 00:14:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [26]:
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load("/opt/spark/work-dir/data/ecommerce_orders.csv")

In [5]:
df.show()

[Stage 1:>                                                          (0 + 1) / 1]

+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+
|order_id|customer_id|product_id|   category| price|quantity|order_status|order_date|payment_method|country|
+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+
|       1|         52|       380|       Home|643.03|     3.0|     Success|2026-02-25| Bank Transfer|     US|
|       2|        448|       120|     Beauty|  39.5|     9.0|     Success|2026-02-25| Bank Transfer| Canada|
|       3|       1781|         4|    Fashion|285.41|     6.0|      FAILED|2024-12-25| Bank Transfer|     US|
|       4|        705|       310|       Home|365.39|     8.0|     success|2023-07-16|        PayPal| Canada|
|       5|        143|        24|      Books|200.37|     5.0|      FAILED|2023-08-30| Bank Transfer|  Japan|
|       6|        334|       190|       Home|835.77|     5.0|     Success|2023-07-01|    Debit Card| Canada|
|       7|        3

In [52]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country: string (nullable = true)



In [53]:
df.groupBy("order_id") \
    .count() \
    .filter(col("count") > 1)  \
    .count()

50

In [54]:
df.select([
    sum(
        when(
            col(c).isNull() | (trim(col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
]).show()

+--------+-----------+----------+--------+-----+--------+------------+----------+--------------+-------+
|order_id|customer_id|product_id|category|price|quantity|order_status|order_date|payment_method|country|
+--------+-----------+----------+--------+-----+--------+------------+----------+--------------+-------+
|       0|          0|         0|       0|    0|     517|           0|         0|             0|      0|
+--------+-----------+----------+--------+-----+--------+------------+----------+--------------+-------+



In [55]:
df.filter(col("quantity").isNull()).count()

517

In [56]:
df_check_date = df.withColumn(
    "parsed_order_date",
    to_date(col("order_date"), "yyyy-MM-dd")
)
df_check_date.filter(
    col("order_date").isNotNull() & col("parsed_order_date").isNull()
).show(truncate=False)

+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-----------------+
|order_id|customer_id|product_id|category   |price |quantity|order_status|order_date|payment_method|country|parsed_order_date|
+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-----------------+
|54      |692        |390       |Sports     |460.15|7.0     |SUCCESS     |bad_date  |Credit Card   |Germany|NULL             |
|81      |60         |499       |Books      |489.86|4.0     |PENDING     |bad_date  |Bank Transfer |US     |NULL             |
|117     |879        |433       |Beauty     |233.9 |10.0    |Success     |bad_date  |PayPal        |US     |NULL             |
|122     |997        |270       |Books      |681.7 |4.0     |failed      |bad_date  |PayPal        |UK     |NULL             |
|145     |1700       |242       |Books      |294.01|8.0     |SUCCESS     |bad_date  |Debit Card    |Vietnam|NUL

In [63]:
df.filter(col("order_date") == "bad_date").count()

138

In [58]:
df.select("order_status").distinct().show()

+------------+
|order_status|
+------------+
|     success|
|      failed|
|     Success|
|     SUCCESS|
|      FAILED|
|     PENDING|
+------------+



Processing Data

In [27]:
df = df.dropDuplicates(["order_id"])

In [60]:
df.count()

5000

In [28]:
df = df.filter(col("quantity").isNotNull())

In [62]:
df.count()

4484

In [29]:
df = df.filter(col("order_date") != "bad_date")

In [65]:
df.count()

4346

In [30]:
df = df.withColumn(
    "order_status",
    lower(trim(col("order_status")))
)

In [31]:
df.select("order_status").distinct().count()

3

In [32]:
numeric_cols = ["order_id", "customer_id", "product_id","quantity"]
for c in numeric_cols:
    print(f"Checking column: {c}")
    
    df.filter(
        col(c).isNotNull() &
        col(c).cast("double").isNull()
    ).select(c).distinct().show(truncate=False)

Checking column: order_id
+--------+
|order_id|
+--------+
+--------+

Checking column: customer_id
+-----------+
|customer_id|
+-----------+
+-----------+

Checking column: product_id
+----------+
|product_id|
+----------+
+----------+

Checking column: quantity
+--------+
|quantity|
+--------+
+--------+



In [69]:
df.write.mode("append").parquet("/opt/spark/work-dir/output/ecommerce_data_clean")

In [16]:
df_par = spark.read \
    .format("parquet") \
    .option("header", "true") \
    .load("/opt/spark/work-dir/output/ecommerce_data_clean")

In [17]:
df_par.show()

[Stage 6:>                                                          (0 + 1) / 1]

+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+
|order_id|customer_id|product_id|   category| price|quantity|order_status|order_date|payment_method|country|
+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+
|       1|         52|       380|       Home|643.03|     3.0|     success|2026-02-25| Bank Transfer|     US|
|      10|        284|       261|     Sports|227.13|     1.0|      failed|2023-12-29| Bank Transfer| Canada|
|     100|         29|       136|     Beauty|792.78|     6.0|     success|2024-07-25|        PayPal|     US|
|    1000|       1435|       319|     Sports|508.21|     4.0|     success|2023-06-27|        PayPal| France|
|    1001|         27|       342|     Sports|150.42|     1.0|     pending|2023-07-12|   Credit Card| Canada|
|    1002|        971|       299|Electronics|358.62|     4.0|     success|2023-03-12|        PayPal|Vietnam|
|    1003|       10

In [18]:
df_par.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country: string (nullable = true)



In [33]:
df_clean = df \
    .withColumn("price",col("price").cast("double")) \
    .withColumn("quantity",col("quantity").cast("int")) \
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd")) 

In [34]:
df_clean.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- country: string (nullable = true)



In [35]:
df_clean.show()

+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+
|order_id|customer_id|product_id|   category| price|quantity|order_status|order_date|payment_method|country|
+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+
|       1|         52|       380|       Home|643.03|       3|     success|2026-02-25| Bank Transfer|     US|
|      10|        284|       261|     Sports|227.13|       1|      failed|2023-12-29| Bank Transfer| Canada|
|     100|         29|       136|     Beauty|792.78|       6|     success|2024-07-25|        PayPal|     US|
|    1000|       1435|       319|     Sports|508.21|       4|     success|2023-06-27|        PayPal| France|
|    1001|         27|       342|     Sports|150.42|       1|     pending|2023-07-12|   Credit Card| Canada|
|    1002|        971|       299|Electronics|358.62|       4|     success|2023-03-12|        PayPal|Vietnam|
|    1003|       10

In [36]:
df_clean.write.mode("append").parquet("/opt/spark/work-dir/output/ecommerce_data")

In [42]:
df = df_clean.withColumn(
    "revenue",
    round(col("price") * col("quantity"),2)
)

In [45]:
df = df.withColumn(
    "order_month",
    date_format(col("order_date"), "yyyy-MM")
)

In [46]:
df.show()

+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-------+-----------+
|order_id|customer_id|product_id|   category| price|quantity|order_status|order_date|payment_method|country|revenue|order_month|
+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-------+-----------+
|       1|         52|       380|       Home|643.03|       3|     success|2026-02-25| Bank Transfer|     US|1929.09|    2026-02|
|      10|        284|       261|     Sports|227.13|       1|      failed|2023-12-29| Bank Transfer| Canada| 227.13|    2023-12|
|     100|         29|       136|     Beauty|792.78|       6|     success|2024-07-25|        PayPal|     US|4756.68|    2024-07|
|    1000|       1435|       319|     Sports|508.21|       4|     success|2023-06-27|        PayPal| France|2032.84|    2023-06|
|    1001|         27|       342|     Sports|150.42|       1|     pending|2023-07-12|   Credit Ca

In [49]:
monthly_revenue = df.groupBy("order_month") \
    .agg(round(sum("revenue"),2).alias("total_revenue"))

In [50]:
monthly_revenue.show()

+-----------+-------------+
|order_month|total_revenue|
+-----------+-------------+
|    2026-05|     131397.7|
|    2024-09|    374839.65|
|    2023-08|    338840.54|
|    2024-02|    298233.05|
|    2023-12|    297069.61|
|    2025-04|    281681.75|
|    2023-11|    324915.69|
|    2025-01|    254914.34|
|    2025-08|    307063.58|
|    2023-07|    226451.06|
|    2024-08|    261175.15|
|    2023-03|    308969.29|
|    2023-10|    328433.72|
|    2026-04|    294250.79|
|    2024-06|    348640.56|
|    2023-02|    244179.42|
|    2023-09|    322822.97|
|    2023-04|    267291.98|
|    2023-05|     327176.9|
|    2024-12|    320778.35|
+-----------+-------------+
only showing top 20 rows



In [52]:
revenue_country = df.groupBy("country").agg(
    round(sum("revenue"),2).alias("revenue_per_country"))

In [53]:
revenue_country.show()

+-------+-------------------+
|country|revenue_per_country|
+-------+-------------------+
|Germany|         1691944.78|
| France|         1879577.22|
|     US|         1748998.53|
|     UK|         1659707.12|
| Canada|         1566020.29|
|  Japan|         1765145.31|
|Vietnam|         1754844.25|
+-------+-------------------+



In [4]:
df = spark.read \
    .format("parquet") \
    .option("header","true") \
    .load("/opt/spark/work-dir/output/ecommerce_data")

In [7]:
df.show()

[Stage 2:>                                                          (0 + 1) / 1]

+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-------+
|order_id|customer_id|product_id|   category| price|quantity|order_status|order_date|payment_method|country|revenue|
+--------+-----------+----------+-----------+------+--------+------------+----------+--------------+-------+-------+
|       1|         52|       380|       Home|643.03|       3|     success|2026-02-25| Bank Transfer|     US|1929.09|
|      10|        284|       261|     Sports|227.13|       1|      failed|2023-12-29| Bank Transfer| Canada| 227.13|
|     100|         29|       136|     Beauty|792.78|       6|     success|2024-07-25|        PayPal|     US|4756.68|
|    1000|       1435|       319|     Sports|508.21|       4|     success|2023-06-27|        PayPal| France|2032.84|
|    1001|         27|       342|     Sports|150.42|       1|     pending|2023-07-12|   Credit Card| Canada| 150.42|
|    1002|        971|       299|Electronics|358.62|       4|   

In [6]:
df = df.withColumn(
    "revenue",
    round(col("price") * col("quantity"),2)
)

In [9]:
revenue_categories = df.groupBy('category').agg(
    round(sum('revenue'),2).alias("revenue_by_category")
)

In [10]:
revenue_categories.show()

[Stage 3:>                                                          (0 + 1) / 1]

+-----------+-------------------+
|   category|revenue_by_category|
+-----------+-------------------+
|       Home|         2063246.14|
|    Fashion|         2002629.13|
|     Sports|         2130138.58|
|Electronics|          1848587.8|
|      Books|         2004399.57|
|     Beauty|         2017236.28|
+-----------+-------------------+

